In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from data import CaseHFL, CaseHFL_Heterogenous
from sklearn.preprocessing import StandardScaler

In [2]:
class MAMLRegression(nn.Module):
    def __init__(self, input_size):
        super(MAMLRegression, self).__init__()

        self.weight = nn.Parameter(torch.Tensor(1, input_size))
        self.bias = nn.Parameter(torch.Tensor(1))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)
        
    def forward(self, x, weights=None):
        # bias + x times weights
        if weights is None:
            return torch.addmm(self.bias, x, self.weight.t())
        
        return torch.addmm(weights['bias'], x, weights['weight'].t())

def maml_update(model, x_train, y_train, x_validation, y_validation, inner_lr, meta_optimizer):
    meta_optimizer.zero_grad()
    meta_loss = 0.0
    
    for i in range(len(x_train)):
        x_support = torch.tensor(x_train[i].T, dtype=torch.float32)
        y_support = torch.tensor(y_train[i], dtype=torch.float32).view(-1, 1)
        x_query = torch.tensor(x_validation[i].T, dtype=torch.float32)
        y_query = torch.tensor(y_validation[i], dtype=torch.float32).view(-1, 1)

        pred_s = model(x_support)
        inner_loss = nn.MSELoss()(pred_s, y_support)
        
        grads = torch.autograd.grad(inner_loss, model.parameters(), create_graph=True)
        
        hypothetical_weights = {}
        for (name, param), grad in zip(model.named_parameters(), grads):
            hypothetical_weights[name] = param - inner_lr * grad
            
        pred_query = model(x_query, weights=hypothetical_weights)
        task_meta_loss = nn.MSELoss()(pred_query, y_query)
        meta_loss += task_meta_loss

    average_meta_loss = meta_loss / len(x_train)
    average_meta_loss.backward()
    meta_optimizer.step()
    
    return average_meta_loss.item()

In [3]:
rng = np.random.default_rng(42)
case_train_test = CaseHFL_Heterogenous(
    rng = rng,
    n = 1000,
    events = 100,
    output_arity = 50,
    sensors = 10,
    noise_std = 0.01,
    heterogeneity_std=1.0
)

case_validation = CaseHFL_Heterogenous(
    rng = rng,
    n = 200,
    events = 100,
    output_arity = 50,
    sensors = 10,
    noise_std = 0.01,
    heterogeneity_std=1.0
)

In [4]:
X_train, Y_train = case_train_test.generate_data()
X_validation, Y_validation = case_validation.generate_data()
X_test, Y_test = case_train_test.generate_data()
scaler_x = StandardScaler()
scaler_y = StandardScaler()


X_train = [scaler_x.fit_transform(x.T).T for x in X_train]
X_validation = [scaler_x.fit_transform(x.T).T for x in X_validation]
X_test = [scaler_x.fit_transform(x.T).T for x in X_test]
Y_train = [scaler_y.fit_transform(y.reshape(-1, 1)).flatten() for y in Y_train]
Y_validation = [scaler_y.fit_transform(y.reshape(-1, 1)).flatten() for y in Y_validation]
Y_test = [scaler_y.fit_transform(y.reshape(-1, 1)).flatten() for y in Y_test]

In [5]:
input_dim = X_train[0].shape[0]
meta_model = MAMLRegression(input_dim)
meta_opt = optim.Adam(meta_model.parameters(), lr=0.001)

for epoch in range(100):
   loss = maml_update(meta_model, X_train, Y_train, X_validation, Y_validation, 0.01, meta_opt)

In [6]:
def evaluate_maml(model, x_support, y_support, x_query, y_query, inner_lr, steps=1):
    model.eval()
    
    x_support = torch.tensor(x_support.T, dtype=torch.float32)
    y_support = torch.tensor(y_support, dtype=torch.float32).view(-1, 1)
    x_query = torch.tensor(x_query.T, dtype=torch.float32)
    y_query = torch.tensor(y_query, dtype=torch.float32).view(-1, 1)

    new_learned_weights = {name: param for name, param in model.named_parameters()}
    
    for _ in range(steps):
        pred_s = model(x_support, weights=new_learned_weights)
        loss_s = nn.MSELoss()(pred_s, y_support)
        
        grads = torch.autograd.grad(loss_s, new_learned_weights.values())
        
        new_learned_weights = {
            name: param - inner_lr * grad 
            for (name, param), grad in zip(new_learned_weights.items(), grads)
        }

    with torch.no_grad():
        pred_q = model(x_query, weights=new_learned_weights)
        query_loss = nn.MSELoss()(pred_q, y_query)
    
    return query_loss.item(), pred_q.numpy()

In [7]:
test_losses = []
all_predictions = []

inner_lr = 0.01 
adaptation_steps = 3 

for i in range(len(X_test)):
    sample_count = X_test[i].shape[1]
    
    split_idx = int(sample_count * 0.2) 
    
    x_support = X_test[i][:, :split_idx]
    y_support = Y_test[i][:split_idx]
    
    x_query = X_test[i][:, split_idx:]
    y_query = Y_test[i][split_idx:]
    
    mse, preds = evaluate_maml(
        meta_model, 
        x_support, 
        y_support, 
        x_query, 
        y_query, 
        inner_lr, 
        steps=adaptation_steps
    )
    
    test_losses.append(mse)
    all_predictions.append(preds)

average_mse = np.mean(test_losses)
print(f"Average Test MSE across {len(X_test)} tasks: {average_mse:.4f}")

Average Test MSE across 10 tasks: 1.2357
